<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_08_transformer_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06_08 - SEQ2ONE - Transformer**

El Transformer es un modelo basado en self-attention que procesa secuencias completas en paralelo, sin recurrencia. En el esquema many-to-one, el modelo recibe una ventana temporal de múltiples pasos (many) y produce una única salida agregada (one), típicamente usando el embedding del último token o un pooling sobre la secuencia.

Su ventaja clave es capturar dependencias de largo alcance de forma eficiente, con alta escalabilidad y estabilidad en el entrenamiento frente a RNN/LSTM.


# **BLOQUE DE EJECUCIÓN COMPLETO**

## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [3]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

window_sizes = [30, 60, 90, 120, 180]
targets = ['delta_60', 'delta_90', 'ret_60', 'ret_90']
splits = ['train', 'valid', 'test']

In [4]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [5]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl'),
 'delta_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_90.pkl'),
 'ret_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_60.pkl'),
 'ret_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_90.pkl')}

## **4. Reproducibilidad**

In [6]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [7]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [8]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


## **6. Carga de data windows**

In [9]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y

In [10]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [11]:
from typing import Any, Dict, Mapping
from pathlib import Path

def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
    splits: tuple[str, ...] = ("train", "valid", "test"),
) -> Dict[str, Any]:
    """
    Carga X/y para los splits solicitados y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path (.npz con X,y)
    scalers_path[target] -> Path (scaler)
    """

    # --------------------------
    # 1) Validaciones base
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    valid_splits = {"train", "valid", "test"}
    splits_set = set(splits)
    unknown = splits_set - valid_splits
    if unknown:
        raise ValueError(f"splits inválidos: {sorted(unknown)}. Usar {sorted(valid_splits)}")

    # Validar que existan los splits solicitados para ese target
    available = set(windows_paths[window_size][target].keys())
    missing = splits_set - available
    if missing:
        raise KeyError(
            f"Faltan splits {sorted(missing)} en windows_paths[{window_size}]['{target}']. "
            f"Disponibles: {sorted(available)}"
        )

    # --------------------------
    # 2) Paths (solo los necesarios)
    # --------------------------
    split_paths: Dict[str, Path] = {sp: windows_paths[window_size][target][sp] for sp in splits}
    scaler_path = scalers_path[target]

    # --------------------------
    # 3) Carga por split
    # --------------------------
    out_splits: Dict[str, Dict[str, Any]] = {}
    out_paths: Dict[str, str] = {}

    for sp, p in split_paths.items():
        X, y = load_npz_windows(p)
        out_splits[sp] = {"X": X, "y": y}
        out_paths[sp] = str(p)

    scaler = load_scaler(scaler_path)
    out_paths["scaler"] = str(scaler_path)

    # --------------------------
    # 4) Inferir horizonte (robusto)
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception as e:
        raise ValueError(f"No se pudo inferir horizon desde target='{target}'. Esperado sufijo '_<int>'") from e

    # --------------------------
    # 5) Retorno
    # --------------------------
    out: Dict[str, Any] = {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": out_paths,
        "scaler": scaler,
    }
    out.update(out_splits)

    return out

In [12]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [13]:
def create_bundles(
    window_size,
    targets: list,
    windows_paths=windows_paths,
    scalers_paths=scalers_paths,
    *,
    flatten_X: bool = False,
    splits: tuple[str, ...] = ("train", "valid", "test"),
    verbose_shapes: bool = True,
    ):
    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
            splits=splits,
        )

        if flatten_X:
            for sp in splits:
                b[sp]["X"] = maybe_flatten_X(b[sp]["X"], flatten=True)

        bundles.append(b)

    if verbose_shapes:
        for b in bundles:
            for sp in splits:
                print(f"H{b['horizon']} {sp.capitalize():<5}:", b[sp]["X"].shape, b[sp]["y"].shape)
            print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [14]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [15]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [16]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [17]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **8. Métricas ML**


In [18]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

## **9. Gestión de dataset de métricas**

In [19]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [20]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

In [21]:
import gc, torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gc.collect()
torch.cuda.empty_cache()

# **DEFINICIÓN DE MODELO**

## **10. Definición del modelo — placeholder**

### **10.1. Modelo Transformer — seq2one**

**Idea básica**

El **Transformer** es un modelo de aprendizaje profundo basado en el
mecanismo de **self-attention**, diseñado para modelar dependencias
temporales **sin recurrencia** y con procesamiento completamente
paralelo de la secuencia de entrada.

En el esquema **many-to-one**, el modelo recibe una **ventana temporal**
de múltiples pasos (many) y produce **un único valor escalar futuro**
(one), asociado al final de la ventana.

A diferencia del MLP, el Transformer **preserva explícitamente la
estructura temporal**, permitiendo que cada instante de la secuencia
atienda a cualquier otro instante según su relevancia para la
predicción final.

Formalmente, el modelo puede representarse como:

$$
\hat{y}_t = g\Big( \text{Pool}\big( \text{TransformerEncoder}(X_t) \big) \Big)
$$

donde:
- $X_t \in \mathbb{R}^{T \times F}$ es la ventana temporal (longitud $T$, $F$ features),
- $\text{TransformerEncoder}(\cdot)$ aplica capas de self-attention y feedforward,
- $\text{Pool}(\cdot)$ es una agregación temporal (último token, mean pooling, etc.),
- $g(\cdot)$ es una capa densa final que produce el target escalar.

---

**Regularización (Transformer)**

**Riesgo:** Alto, debido a la elevada capacidad del modelo.

El Transformer incorpora **regularización parcial de forma intrínseca**,
pero requiere control explícito:

- **Dropout (intrínseco):**
  - Aplicado en self-attention y capas feedforward.
- **Early stopping:**
  - Fundamental para evitar sobreajuste.
- **Dimensión del embedding controlada:**
  - Evita representaciones excesivamente complejas.
- **Número limitado de capas encoder:**
  - 1–3 capas en escenarios de datos financieros.
- **Weight decay (opcional):**
  - Refuerza la estabilidad del entrenamiento.

---

**Por qué el Transformer es relevante en este proyecto**

- Capacidad para capturar:
  - dependencias **de largo alcance**,
  - relaciones temporales no locales.
- Adecuado para:
  - ventanas largas (60–90 minutos),
  - múltiples indicadores técnicos simultáneos.
- Entrenamiento:
  - paralelo y estable,
  - más escalable que LSTM/GRU.

El Transformer representa el **primer modelo plenamente atencional**
del pipeline, sirviendo como referencia frente a arquitecturas
secuenciales (LSTM, GRU) y convolucionales (TCN).

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Número de capas encoder: 1–2
- Dimensión del embedding: moderada (32–64)
- Número de cabezas de atención: 2–4
- Dropout: activado
- Pooling temporal: último token o mean pooling
- Early stopping: activado
- **Sin tuning exhaustivo** (optimización posterior)

El ajuste fino de profundidad, atención y regularización se aborda en
etapas posteriores del proyecto.


### **10.2. Imports y “seed” (base reproducible)**

In [22]:
# Paso 1: imports básicos + reproducibilidad (sin tqdm)
import os
import json
import random
from pathlib import Path
from typing import Dict, Any, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [23]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # reproducibilidad (puede bajar performance, pero estable)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


### **10.3. TensorDataset + DataLoader (PyTorch)**

In [24]:
# Paso 3: TensorDataset y DataLoaders para H60 y H90 (sin tqdm)

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def make_loaders_from_bundle_3d(
    bundle: dict,
    *,
    seq_len: int | None = None,
    n_features: int | None = None,
    batch_size: int = 1024,
    num_workers: int = 2,
) -> dict:
    """
    Crea loaders train/valid/test para TRANSFORMER many-to-one.

    Espera:
      - X: (n, seq_len, n_features)  (ya 3D)
      - y: (n,) o (n,1)  -> (n,1)

    Si seq_len/n_features se pasan, valida consistencia.
    """
    loaders = {}

    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)

        if X.ndim != 3:
            raise ValueError(
                f"[{split}] Se esperaba X 3D (n, seq_len, n_features). "
                f"Recibido shape={X.shape} (ndim={X.ndim})."
            )

        n, sl, nf = X.shape

        if seq_len is not None and sl != int(seq_len):
            raise ValueError(f"[{split}] seq_len esperado={seq_len}, recibido={sl}. shape={X.shape}")

        if n_features is not None and nf != int(n_features):
            raise ValueError(f"[{split}] n_features esperado={n_features}, recibido={nf}. shape={X.shape}")

        if y.shape[0] != n:
            raise ValueError(f"[{split}] X e y no alinean: X n={n}, y n={y.shape[0]}.")

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )

    return loaders

### **10.4. Definición de modelo Transformer (many-to-one)**

In [25]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, d_model)
        L = x.size(1)
        x = x + self.pe[:, :L, :]
        return self.dropout(x)

class TransformerManyToOne(nn.Module):
    def __init__(
        self,
        *,
        n_features: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dim_ff: int = 128,
        dropout: float = 0.1,
        pooling: str = "mean",  # "mean" o "last"
    ):
        super().__init__()
        self.pooling = pooling

        self.in_proj = nn.Linear(n_features, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,   # (B, L, D)
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, F)
        z = self.in_proj(x)          # (B, L, D)
        z = self.pos_enc(z)          # (B, L, D)
        z = self.encoder(z)          # (B, L, D)

        if self.pooling == "last":
            pooled = z[:, -1, :]     # (B, D)
        else:
            pooled = z.mean(dim=1)   # (B, D)

        out = self.head(pooled)      # (B, 1)
        return out


Esta implementación usa:
- Proyección: 20 → d_model
- Positional encoding (sin metadata)
- TransformerEncoder
- Pooling: mean sobre el tiempo
- Head: salida escalar

In [26]:
# ============================================================
# 2) Modelo: factory (misma idea que antes)
# ============================================================
def make_transformer_model(*, n_features: int, device: torch.device) -> nn.Module:
    model = TransformerManyToOne(
        n_features=n_features,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_ff=128,
        dropout=0.1,
        pooling="mean",
    ).to(device)
    return model


In [27]:
# ------------------------------------------------------------
# Modelos independientes (H60 y H90)
# IMPORTANTE: n_features debe coincidir con tu X 3D: (B, L, F)
# ------------------------------------------------------------
n_features = 36

In [28]:
# ------------------------------------------------------------
# Smoke test robusto (por horizonte)
# ------------------------------------------------------------
def smoke_test(model, loader, device, name: str):
    model.eval()
    xb, yb = next(iter(loader))
    xb = xb.to(device)
    yb = yb.to(device)

    with torch.no_grad():
        out = model(xb)

    print(f"[{name}] X:", tuple(xb.shape), xb.dtype)
    print(f"[{name}] y:", tuple(yb.shape), yb.dtype)
    print(f"[{name}] out:", tuple(out.shape), out.dtype)

    assert xb.ndim == 3, "X debe ser 3D: (B, L, F)"
    assert out.ndim == 2 and out.shape[1] == 1, "Salida debe ser (B, 1)"
    assert yb.ndim in (1, 2), "y debe ser (B,) o (B,1)"

### **10.5. Definición de loss, optimizer y funciones de train / eval (sin tqdm)**

In [29]:
# ============================================================
# loss, optimizer y funciones de entrenamiento / evaluación
# (preparado para múltiples targets / horizontes / window_size)
# ============================================================

import torch
import torch.nn as nn
from typing import Optional


# ------------------------------------------------------------
# Factory: Loss (regresión)
# ------------------------------------------------------------
def make_criterion() -> nn.Module:
    return nn.MSELoss()


# ------------------------------------------------------------
# Factory: Optimizer (uno por corrida/modelo)
# ------------------------------------------------------------
def make_optimizer(
    model: nn.Module,
    *,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
) -> torch.optim.Optimizer:
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


# ------------------------------------------------------------
# Función de entrenamiento (1 epoch)
# ------------------------------------------------------------
def train_one_epoch(
    model: nn.Module,
    loader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    *,
    clip_grad_norm: Optional[float] = None,
) -> float:
    model.train()
    total_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)

        y_hat = model(xb)
        loss = criterion(y_hat, yb)

        loss.backward()

        if clip_grad_norm is not None and clip_grad_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)

        optimizer.step()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n_samples += bs

    return total_loss / max(n_samples, 1)


# ------------------------------------------------------------
# Función de evaluación (1 epoch)
# ------------------------------------------------------------
@torch.no_grad()
def eval_one_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    model.eval()
    total_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        y_hat = model(xb)
        loss = criterion(y_hat, yb)

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n_samples += bs

    return total_loss / max(n_samples, 1)


# ------------------------------------------------------------
# Ejemplo de uso (por corrida / por horizonte)
# ------------------------------------------------------------
# criterion = make_criterion()
# optimizer_60 = make_optimizer(model_60, lr=1e-3, weight_decay=1e-4)
# train_loss = train_one_epoch(model_60, loaders_60["train"], optimizer_60, criterion, device, clip_grad_norm=1.0)
# valid_loss = eval_one_epoch(model_60, loaders_60["valid"], criterion, device)



### **10.6. Loop de entrenamiento completo con early stopping**



In [30]:
import math
import torch
import torch.nn as nn
from typing import Optional, Dict, Any


def fit_one_run(
    *,
    model: nn.Module,
    train_loader,
    valid_loader,
    device: torch.device,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    save_best: bool = True,
    best_path: Optional[str] = None,
    clip_grad_norm: Optional[float] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Entrena 1 modelo (1 target/horizonte/window_size) con early stopping en VALID.
    Retorna: history + best info. Deja el modelo restaurado al mejor estado si save_best=True.
    """
    criterion = make_criterion()
    optimizer = make_optimizer(model, lr=lr, weight_decay=weight_decay)

    best_val = math.inf
    best_epoch = -1
    patience_left = patience

    history = {"train_loss": [], "valid_loss": []}
    best_state = None

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, device, clip_grad_norm=clip_grad_norm
        )
        val_loss = eval_one_epoch(
            model, valid_loader, criterion, device
        )

        history["train_loss"].append(float(train_loss))
        history["valid_loss"].append(float(val_loss))

        improved = (best_val - val_loss) > min_delta
        if improved:
            best_val = float(val_loss)
            best_epoch = epoch
            patience_left = patience

            if save_best:
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                if best_path is not None:
                    torch.save(model.state_dict(), best_path)
        else:
            patience_left -= 1

        if verbose:
            print(
                f"epoch {epoch:02d} | "
                f"train_loss={train_loss:.6f} | "
                f"valid_loss={val_loss:.6f} | "
                f"patience_left={patience_left}"
            )

        if patience_left <= 0:
            if verbose:
                print(f"Early stopping: best_valid_loss={best_val:.6f} at epoch {best_epoch}")
            break

    if save_best and best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
        if verbose:
            print(f"Modelo restaurado: epoch {best_epoch} | best_valid_loss={best_val:.6f}")

    return {
        "best_valid_loss": best_val,
        "best_epoch": best_epoch,
        "epochs_ran": epoch,
        "history": history,
    }


### **10.7. Predicciones Transformer**


Función de predicción (seq2one) -> y_true, y_pred

In [31]:
import numpy as np
import torch

@torch.no_grad()
def predict_seq2one(
    model,
    loader,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()

    y_true_list = []
    y_pred_list = []

    for xb, yb in loader:
        xb = xb.to(device)

        y_hat = model(xb).detach().cpu().numpy()   # (B,1) típico
        y_true = yb.detach().cpu().numpy()         # (B,1) o (B,)

        y_pred_list.append(y_hat)
        y_true_list.append(y_true)

    y_true_all = np.concatenate(y_true_list, axis=0)
    y_pred_all = np.concatenate(y_pred_list, axis=0)

    return y_true_all, y_pred_all

In [32]:
def get_metrics_torch_from_loaders(
    loaders: dict,
    model,
    *,
    device: torch.device,
    predict_loader_fn=predict_seq2one,
    compute_r2: bool = True,
) -> tuple[dict, dict]:
    """
    Calcula métricas valid/test para modelos seq2one usando DataLoaders.
    Espera loaders con keys: 'valid' y 'test'.
    predict_loader_fn debe devolver (y_true_all, y_pred_all).
    """

    # -------- VALID --------
    y_true_valid, y_pred_valid = predict_loader_fn(model, loaders["valid"], device=device)
    y_true_valid = np.asarray(y_true_valid).reshape(-1)
    y_pred_valid = np.asarray(y_pred_valid).reshape(-1)

    metrics_valid = compute_seq2one_metrics(y_true_valid, y_pred_valid, compute_r2=compute_r2)

    # -------- TEST --------
    y_true_test, y_pred_test = predict_loader_fn(model, loaders["test"], device=device)
    y_true_test = np.asarray(y_true_test).reshape(-1)
    y_pred_test = np.asarray(y_pred_test).reshape(-1)

    metrics_test = compute_seq2one_metrics(y_true_test, y_pred_test, compute_r2=compute_r2)

    return metrics_valid, metrics_test

### **11 Funciones de intregación**

### **11.1. Función `train_transformer`**

In [33]:
from typing import Any, Dict, Tuple
import torch
import torch.nn as nn

def train_transformer(
    loaders: dict,
    *,
    n_features: int,
    device: torch.device,

    # ---- hiperparámetros Transformer ----
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 128,
    dropout: float = 0.1,
    pooling: str = "mean",

    # ---- optim / early stopping ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    clip_grad_norm: float | None = 1.0,
    use_scheduler: bool = False,   # por ahora lo ignoramos (para no mezclar pasos)
    save_best: bool = True,
    best_path: str | None = None,
    verbose: bool = True,
) -> Tuple[nn.Module, Dict[str, Any], Dict, Dict]:
    """
    Entrena 1 Transformer (1 target/horizonte/window_size) y devuelve:
      (model, hist, metrics_valid, metrics_test)
    """
    # 1) modelo
    model = TransformerManyToOne(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_ff=dim_ff,
        dropout=dropout,
        pooling=pooling,
    ).to(device)

    # 2) fit (early stopping en VALID)
    hist = fit_one_run(
        model=model,
        train_loader=loaders["train"],
        valid_loader=loaders["valid"],
        device=device,
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=max_epochs,
        patience=patience,
        min_delta=min_delta,
        save_best=save_best,
        best_path=best_path,
        clip_grad_norm=clip_grad_norm,
        verbose=verbose,
    )

    # 3) métricas VALID/TEST usando loaders
    metrics_valid, metrics_test = get_metrics_torch_from_loaders(
        {"valid": loaders["valid"], "test": loaders["test"]},
        model,
        device=device,
        predict_loader_fn=predict_seq2one,
        compute_r2=True,
    )

    return model, hist, metrics_valid, metrics_test


### **11.2. Función `run_transformer`**

In [34]:
import pandas as pd
import torch
import gc
import time

def _ts():
    return time.strftime("%H:%M:%S")


def run_transformer(
    window_size: int,
    *,
    n_features: int = 36,
    batch_size_train: int = 4096,
    batch_size_pred: int = 32768,

    # ---- hiperparámetros Transformer ----
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 128,
    dropout: float = 0.1,
    pooling: str = "mean",

    # ---- optim ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,

    verbose: bool = True,
):
    L = int(window_size)

    if verbose:
        print("\n" + "=" * 80)
        print(
            f"[{_ts()}] TRANSFORMER | SEQ2ONE | WINDOW_SIZE=L{L} | (L,F)=({L},{n_features}) "
            f"| d_model={d_model} | head={nhead} | layers={num_layers} | ff={dim_ff} | do={dropout} "
            f"| wd={weight_decay}"
        )
        print("=" * 80)

    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]
    rows = []
    t_global = time.perf_counter()

    for i, target in enumerate(targets, start=1):
        t_target = time.perf_counter()

        if verbose:
            print(f"\n[{_ts()}] [{i}/{len(targets)}] START target='{target}' | L{L}")

        bundle = None
        loaders = None
        model = None
        hist = None
        metrics_valid = None
        metrics_test = None

        try:
            # -------------------------
            # BUILD BUNDLE (3D)
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [BUILD] Creando bundle (flatten_X=False) ...")
            t0 = time.perf_counter()

            (bundle,) = create_bundles(
                window_size=L,
                targets=[target],
                windows_paths=windows_paths,
                scalers_paths=scalers_paths,
                flatten_X=False,  # Transformer necesita 3D
            )

            if verbose:
                dt = time.perf_counter() - t0
                try:
                    xshape = bundle["train"]["X"].shape
                    yshape = bundle["train"]["y"].shape
                    print(f"[{_ts()}]   [BUILD] OK | train X={xshape} y={yshape} | dt={dt:.2f}s")
                except Exception:
                    print(f"[{_ts()}]   [BUILD] OK | dt={dt:.2f}s")

            # -------------------------
            # LOADERS (train)
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [LOADERS] Creando DataLoaders (3D) ...")
            t0 = time.perf_counter()

            loaders = make_loaders_from_bundle_3d(
                bundle,
                seq_len=L,
                n_features=n_features,
                batch_size=batch_size_train,
            )

            if verbose:
                dt = time.perf_counter() - t0
                try:
                    ntr = len(loaders["train"].dataset)
                    nva = len(loaders["valid"].dataset)
                    nte = len(loaders["test"].dataset)
                    print(f"[{_ts()}]   [LOADERS] OK | n(train/valid/test)=({ntr}/{nva}/{nte}) | dt={dt:.2f}s")
                except Exception:
                    print(f"[{_ts()}]   [LOADERS] OK | dt={dt:.2f}s")

            # -------------------------
            # TRAIN
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [TRAIN] Iniciando entrenamiento ...")
            t0 = time.perf_counter()

            model, hist, metrics_valid, metrics_test = train_transformer(
                loaders,
                n_features=n_features,
                device=device,
                d_model=64,
                nhead=4,
                num_layers=2,
                dim_ff=128,
                dropout=0.1,
                pooling="mean",
                lr=1e-3,
                weight_decay=1e-4,
                max_epochs=30,
                patience=5,
                clip_grad_norm=1.0,
                verbose=verbose,
            )

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [TRAIN] FIN entrenamiento | dt={dt:.2f}s")

            # liberar TRAIN (opcional, igual que GRU)
            if verbose:
                print(f"[{_ts()}]   [MEM] Liberando bundle['train'] y gc.collect() ...")
            del bundle["train"]
            gc.collect()

            # -------------------------
            # PRED LOADERS (valid/test) con batch grande
            # -------------------------
            # Creamos loaders solo para valid/test con batch_size_pred
            if verbose:
                print(f"[{_ts()}]   [PRED] Preparando loaders valid/test (batch_size_pred={batch_size_pred}) ...")
            t0 = time.perf_counter()

            pred_bundle = {
                "valid": bundle["valid"],
                "test": bundle["test"],
                # dummy para cumplir interfaz sin usar train
                "train": {"X": bundle["valid"]["X"][:1], "y": bundle["valid"]["y"][:1]},
            }
            pred_loaders = make_loaders_from_bundle_3d(
                pred_bundle,
                seq_len=L,
                n_features=n_features,
                batch_size=batch_size_pred,
                num_workers=0,  # pred suele ser más estable con 0
            )

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [PRED] Loaders OK | dt={dt:.2f}s")

            # -------------------------
            # METRICS (valid/test) usando loaders
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [METRICS] Calculando métricas (valid/test) ...")
            t0 = time.perf_counter()

            metrics_valid, metrics_test = get_metrics_torch_from_loaders(
                {"valid": pred_loaders["valid"], "test": pred_loaders["test"]},
                model,
                device=device,
                predict_loader_fn=predict_seq2one,
                compute_r2=True,
            )

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [METRICS] OK (valid/test) | dt={dt:.2f}s")

            # -------------------------
            # DF APPEND
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [DF] Agregando filas a la tabla ...")
            t0 = time.perf_counter()

            df_v = metrics_to_df(
                metrics_valid,
                model="transformer",
                split="valid",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )

            df_t = metrics_to_df(
                metrics_test,
                model="transformer",
                split="test",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )

            for df_ in (df_v, df_t):
                df_["d_model"] = d_model
                df_["nhead"] = nhead
                df_["num_layers"] = num_layers
                df_["dim_ff"] = dim_ff
                df_["dropout"] = dropout
                df_["pooling"] = pooling
                df_["lr"] = lr
                df_["weight_decay"] = weight_decay
                if isinstance(hist, dict):
                    df_["best_valid_mse"] = hist.get("best_valid_mse")
                    df_["epochs_ran"] = hist.get("epochs_ran")
                    df_["final_lr"] = hist.get("final_lr")

            rows.append(df_v)
            rows.append(df_t)

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [DF] OK | dt={dt:.2f}s")

            if verbose:
                dt_target = time.perf_counter() - t_target
                print(f"[{_ts()}] [{i}/{len(targets)}] DONE target='{target}' | dt_total={dt_target:.2f}s")

        finally:
            if verbose:
                print(f"[{_ts()}]   [CLEAN] Liberando objetos ...")

            bundle = loaders = model = hist = metrics_valid = metrics_test = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # -------------------------
    # FINAL DF
    # -------------------------
    if verbose:
        print(f"\n[{_ts()}] [FINAL] Concatenando resultados ...")
    t0 = time.perf_counter()

    df_transformer_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        dt = time.perf_counter() - t0
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [FINAL] OK | rows={len(df_transformer_metrics)} | dt_concat={dt:.2f}s | dt_total={dt_all:.2f}s")
        print(df_transformer_metrics[["window_size", "target", "split", "horizon_min", "model"]]
              .drop_duplicates()
              .to_string(index=False))

    return df_transformer_metrics


### **11.3. Función `train_transformer_row`**

In [35]:
import math
import torch
import torch.nn as nn
from typing import Any, Dict, List

def train_transformer_row(
    *,
    loaders: dict,
    window_size: int,
    model_name: str,
    n_features: int = 36,
    device: torch.device,

    # ---- hiperparámetros Transformer ----
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 128,
    dropout: float = 0.1,
    pooling: str = "mean",

    # ---- optim / early stopping ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    clip_grad_norm: float | None = 1.0,
    use_scheduler: bool = False,   # por ahora no lo usamos
    verbose: bool = True,

    # ---- metadata extra (target/horizon/etc.) ----
    target: str | None = None,
    horizon_min: int | None = None,
) -> List[Dict[str, Any]]:
    """
    Entrena 1 corrida y retorna 2 rows (valid/test) como dicts listos para DataFrame.
    Requiere que existan: fit_one_run, get_metrics_torch_from_loaders, predict_seq2one, metrics_to_df.
    """

    # 1) crear modelo
    model = TransformerManyToOne(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_ff=dim_ff,
        dropout=dropout,
        pooling=pooling,
    ).to(device)

    # 2) entrenar (early stopping en VALID)
    hist = fit_one_run(
        model=model,
        train_loader=loaders["train"],
        valid_loader=loaders["valid"],
        device=device,
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=max_epochs,
        patience=patience,
        min_delta=min_delta,
        save_best=True,
        best_path=None,
        clip_grad_norm=clip_grad_norm,
        verbose=verbose,
    )

    # 3) métricas VALID/TEST (usando loaders)
    metrics_valid, metrics_test = get_metrics_torch_from_loaders(
        {"valid": loaders["valid"], "test": loaders["test"]},
        model,
        device=device,
        predict_loader_fn=predict_seq2one,
        compute_r2=True,
    )

    # 4) pasar a DF (como en GRU) y luego a dict rows
    df_v = metrics_to_df(
        metrics_valid,
        model=model_name,
        split="valid",
        horizon=horizon_min if horizon_min is not None else loaders.get("horizon"),
        window_size=window_size,
        target=target if target is not None else loaders.get("target"),
    )

    df_t = metrics_to_df(
        metrics_test,
        model=model_name,
        split="test",
        horizon=horizon_min if horizon_min is not None else loaders.get("horizon"),
        window_size=window_size,
        target=target if target is not None else loaders.get("target"),
    )

    # 5) anexar hiperparámetros + hist (nombres consistentes con GRU)
    for df_ in (df_v, df_t):
        df_["d_model"] = d_model
        df_["nhead"] = nhead
        df_["num_layers"] = num_layers
        df_["dim_ff"] = dim_ff
        df_["dropout"] = dropout
        df_["pooling"] = pooling
        df_["lr"] = lr
        df_["weight_decay"] = weight_decay

        df_["best_valid_mse"] = hist.get("best_valid_loss")  # (es MSELoss)
        df_["epochs_ran"] = hist.get("epochs_ran")
        df_["best_epoch"] = hist.get("best_epoch")

    # liberar un poco
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return [df_v.iloc[0].to_dict(), df_t.iloc[0].to_dict()]


### **11.4. Función `run_transformer_incremental`**

In [36]:
from pathlib import Path
import pandas as pd

def run_transformer_incremental(
    *,
    window_sizes: list[int],

    # ---- hiperparámetros Transformer ----
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 128,
    dropout: float = 0.1,
    pooling: str = "mean",

    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,

    # ---- data ----
    n_features: int = 36,
    batch_size_train: int = 4096,
    batch_size_pred: int = 32768,

    # ---- persistencia ----
    name: str = "transformer",
    verbose: bool = True,
) -> pd.DataFrame:

    # ------------------------------------------------------------
    # Cargar histórico desde su carpeta Drive (si existe)
    # ------------------------------------------------------------
    metrics_dir = Path("/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics")
    metrics_path = metrics_dir / f"seq2one_{name}_metrics.parquet"

    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    expected_targets = {"delta_60", "delta_90", "ret_60", "ret_90"}
    expected_splits = {"valid", "test"}

    dfs_new = []

    for L in window_sizes:
        L = int(L)

        # ---- Skip robusto ----
        if not df_hist.empty:
            dfL = df_hist[(df_hist["model"] == name) & (df_hist["window_size"] == L)]
            done_targets = set(dfL["target"].unique()) if not dfL.empty else set()
            done_splits = set(dfL["split"].unique()) if not dfL.empty else set()

            if expected_targets.issubset(done_targets) and expected_splits.issubset(done_splits):
                if verbose:
                    print(f"[SKIP] {name} L={L} ya existe completo en Drive")
                continue

        # ---- Ejecutar entrenamiento completo para L ----
        df_L = run_transformer(
            window_size=L,
            n_features=n_features,
            batch_size_train=batch_size_train,
            batch_size_pred=batch_size_pred,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            dim_ff=dim_ff,
            dropout=dropout,
            pooling=pooling,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            clip_grad_norm=clip_grad_norm,
            use_scheduler=use_scheduler,
            verbose=verbose,
        )

        df_L["model"] = name
        dfs_new.append(df_L)

        # actualizar histórico en memoria
        if df_hist.empty:
            df_hist = df_L.copy()
        else:
            df_hist = pd.concat([df_hist, df_L], ignore_index=True)

        # ---- Guardar usando SU función ----
        save_seq2one_metrics(df_hist, name=name)

    return df_hist.sort_values(
        ["window_size", "target", "split", "horizon_min", "model"]
    ).reset_index(drop=True)


## **12. Aplicación**

In [37]:
df_transformer_all_sizes = run_transformer_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_ff=128,
    dropout=0.1,
    pooling="mean",
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=30,
    patience=5,
    clip_grad_norm=1.0,
    use_scheduler=True,
    name="transformer",
    verbose=True,
)


[12:47:12] TRANSFORMER | SEQ2ONE | WINDOW_SIZE=L30 | (L,F)=(30,36) | d_model=64 | head=4 | layers=2 | ff=128 | do=0.1 | wd=0.0001

[12:47:12] [1/4] START target='delta_60' | L30
[12:47:12]   [BUILD] Creando bundle (flatten_X=False) ...
H60 Train: (463872, 30, 36) (463872,)
H60 Valid: (99328, 30, 36) (99328,)
H60 Test : (99840, 30, 36) (99840,)
Scaler H60: StandardScaler
[12:47:22]   [BUILD] OK | train X=(463872, 30, 36) y=(463872,) | dt=10.26s
[12:47:22]   [LOADERS] Creando DataLoaders (3D) ...
[12:47:25]   [LOADERS] OK | n(train/valid/test)=(463872/99328/99840) | dt=2.57s
[12:47:25]   [TRAIN] Iniciando entrenamiento ...


/tmp/ipykernel_823/3848978488.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


epoch 01 | train_loss=2815.933228 | valid_loss=2532.223776 | patience_left=5
epoch 02 | train_loss=2784.728068 | valid_loss=2589.361467 | patience_left=4
epoch 03 | train_loss=2749.875665 | valid_loss=2641.901355 | patience_left=3
epoch 04 | train_loss=2704.382851 | valid_loss=2651.432703 | patience_left=2
epoch 05 | train_loss=2651.588001 | valid_loss=2781.015001 | patience_left=1
epoch 06 | train_loss=2591.801029 | valid_loss=3086.214506 | patience_left=0
Early stopping: best_valid_loss=2532.223776 at epoch 1
Modelo restaurado: epoch 1 | best_valid_loss=2532.223776
[12:48:22]   [TRAIN] FIN entrenamiento | dt=57.26s
[12:48:22]   [MEM] Liberando bundle['train'] y gc.collect() ...
[12:48:22]   [PRED] Preparando loaders valid/test (batch_size_pred=32768) ...
[12:48:23]   [PRED] Loaders OK | dt=0.77s
[12:48:23]   [METRICS] Calculando métricas (valid/test) ...
[12:48:26]   [METRICS] OK (valid/test) | dt=3.21s
[12:48:26]   [DF] Agregando filas a la tabla ...
[12:48:26]   [DF] OK | dt=0.01s


/tmp/ipykernel_823/3848978488.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


epoch 01 | train_loss=4318.131385 | valid_loss=4036.484878 | patience_left=5
epoch 02 | train_loss=4272.800639 | valid_loss=4225.987118 | patience_left=4
epoch 03 | train_loss=4214.150673 | valid_loss=4349.664432 | patience_left=3
epoch 04 | train_loss=4147.133893 | valid_loss=4756.729608 | patience_left=2
epoch 05 | train_loss=4061.846900 | valid_loss=6700.895216 | patience_left=1
epoch 06 | train_loss=3975.754585 | valid_loss=7622.813748 | patience_left=0
Early stopping: best_valid_loss=4036.484878 at epoch 1
Modelo restaurado: epoch 1 | best_valid_loss=4036.484878
[12:49:28]   [TRAIN] FIN entrenamiento | dt=51.03s
[12:49:28]   [MEM] Liberando bundle['train'] y gc.collect() ...
[12:49:28]   [PRED] Preparando loaders valid/test (batch_size_pred=32768) ...
[12:49:29]   [PRED] Loaders OK | dt=0.78s
[12:49:29]   [METRICS] Calculando métricas (valid/test) ...
[12:49:32]   [METRICS] OK (valid/test) | dt=3.10s
[12:49:32]   [DF] Agregando filas a la tabla ...
[12:49:32]   [DF] OK | dt=0.01s


...

epoch 01 | train_loss=5014.193332 | valid_loss=4550.655009 | patience_left=5
epoch 02 | train_loss=4928.754910 | valid_loss=4932.145178 | patience_left=4
epoch 03 | train_loss=4721.244154 | valid_loss=5320.929587 | patience_left=3
epoch 04 | train_loss=4469.977537 | valid_loss=5310.871840 | patience_left=2
epoch 05 | train_loss=4231.192667 | valid_loss=6020.538373 | patience_left=1
epoch 06 | train_loss=4008.360083 | valid_loss=6786.741306 | patience_left=0
Early stopping: best_valid_loss=4550.655009 at epoch 1
Modelo restaurado: epoch 1 | best_valid_loss=4550.655009
[13:59:36]   [TRAIN] FIN entrenamiento | dt=248.92s
[13:59:36]   [MEM] Liberando bundle['train'] y gc.collect() ...
[13:59:36]   [PRED] Preparando loaders valid/test (batch_size_pred=32768) ...
[13:59:39]   [PRED] Loaders OK | dt=3.30s
[13:59:39]   [METRICS] Calculando métricas (valid/test) ...
[13:59:46]   [METRICS] OK (valid/test) | dt=7.07s
[13:59:46]   [DF] Agregando filas a la tabla ...
[13:59:46]   [DF] OK | dt=0.01s

/tmp/ipykernel_823/3848978488.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


epoch 01 | train_loss=0.029173 | valid_loss=0.000141 | patience_left=5
epoch 02 | train_loss=0.000158 | valid_loss=0.000107 | patience_left=5
epoch 03 | train_loss=0.000116 | valid_loss=0.000143 | patience_left=4
epoch 04 | train_loss=0.000096 | valid_loss=0.000197 | patience_left=3
epoch 05 | train_loss=0.000084 | valid_loss=0.000162 | patience_left=2
epoch 06 | train_loss=0.000075 | valid_loss=0.000167 | patience_left=1
epoch 07 | train_loss=0.000068 | valid_loss=0.000151 | patience_left=0
Early stopping: best_valid_loss=0.000107 at epoch 2
Modelo restaurado: epoch 2 | best_valid_loss=0.000107
[14:05:04]   [TRAIN] FIN entrenamiento | dt=289.60s
[14:05:04]   [MEM] Liberando bundle['train'] y gc.collect() ...
[14:05:04]   [PRED] Preparando loaders valid/test (batch_size_pred=32768) ...
[14:05:08]   [PRED] Loaders OK | dt=3.29s
[14:05:08]   [METRICS] Calculando métricas (valid/test) ...
[14:05:15]   [METRICS] OK (valid/test) | dt=7.15s
[14:05:15]   [DF] Agregando filas a la tabla ...
[1

/tmp/ipykernel_823/3848978488.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


epoch 01 | train_loss=0.006677 | valid_loss=0.000065 | patience_left=5
epoch 02 | train_loss=0.000209 | valid_loss=0.000060 | patience_left=5
epoch 03 | train_loss=0.000167 | valid_loss=0.000035 | patience_left=5
epoch 04 | train_loss=0.000139 | valid_loss=0.000032 | patience_left=5
epoch 05 | train_loss=0.000125 | valid_loss=0.000098 | patience_left=4
epoch 06 | train_loss=0.000126 | valid_loss=0.000026 | patience_left=5
epoch 07 | train_loss=0.000121 | valid_loss=0.000063 | patience_left=4
epoch 08 | train_loss=0.000107 | valid_loss=0.000061 | patience_left=3
epoch 09 | train_loss=0.000106 | valid_loss=0.000041 | patience_left=2
epoch 10 | train_loss=0.000091 | valid_loss=0.000040 | patience_left=1
epoch 11 | train_loss=0.000083 | valid_loss=0.000174 | patience_left=0
Early stopping: best_valid_loss=0.000026 at epoch 6
Modelo restaurado: epoch 6 | best_valid_loss=0.000026
[14:13:16]   [TRAIN] FIN entrenamiento | dt=452.62s
[14:13:16]   [MEM] Liberando bundle['train'] y gc.collect() .

In [38]:
save_seq2one_metrics(
    df_transformer_all_sizes,
    name="transformer"
)

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/seq2one_transformer_metrics.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/seq2one_transformer_metrics.parquet')

In [39]:
df_transformer_all_sizes

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,d_model,nhead,num_layers,dim_ff,dropout,pooling,lr,weight_decay,best_valid_mse,epochs_ran,final_lr
0,transformer,test,30,delta_60,60,53.925824,83.363054,-0.001194,0.473361,64,4,2,128,0.1,mean,0.001,0.0001,None,6,None
1,transformer,valid,30,delta_60,60,35.489670,50.321206,-0.004466,0.494049,64,4,2,128,0.1,mean,0.001,0.0001,None,6,None
2,transformer,test,30,delta_90,90,68.869327,104.479113,-0.009455,0.474374,64,4,2,128,0.1,mean,0.001,0.0001,None,6,None
3,transformer,valid,30,delta_90,90,45.277670,63.533336,-0.022286,0.477252,64,4,2,128,0.1,mean,0.001,0.0001,None,6,None
4,transformer,test,30,ret_60,60,0.003270,0.005726,-0.788453,0.507247,64,4,2,128,0.1,mean,0.001,0.0001,None,15,None
5,transformer,valid,30,ret_60,60,0.002312,0.003181,-0.312625,0.514330,64,4,2,128,0.1,mean,0.001,0.0001,None,15,None
6,transformer,test,30,ret_90,90,0.003605,0.007266,-0.843234,0.494261,64,4,2,128,0.1,mean,0.001,0.0001,None,21,None
7,transformer,valid,30,ret_90,90,0.002602,0.003616,-0.083587,0.524706,64,4,2,128,0.1,mean,0.001,0.0001,None,21,None
8,transformer,test,60,delta_60,60,55.493872,85.245774,0.000877,0.475550,64,4,2,128,0.1,mean,0.001,0.0001,None,6,None
9,transformer,valid,60,delta_60,60,36.866469,51.640016,-0.007466,0.472685,64,4,2,128,0.1,mean,0.001,0.0001,None,6,None
